In [14]:
from langchain_community.vectorstores import Cassandra
from langchain_classic.indexes.vectorstore import VectorStoreIndexWrapper

from langchain_ollama import OllamaEmbeddings, OllamaLLM

E:\pip-temp\ipykernel_19260\2535086440.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Cassandra


In [2]:
## Support or dataset retrieval with Hugging Face
from datasets import load_dataset

## With CassIO , The engine powering the Astro DB : in integration in Langchain,
# you will also intitialize the DB connection:

import cassio

e:\agentAI\agentAI\.venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from PyPDF2 import PdfReader

In [6]:
pdfreader = PdfReader('LLM.pdf')
pdfreader

In [7]:
from typing_extensions import Concatenate
raw_text=''
for i , page in enumerate(pdfreader.pages):
    content = page.extract_text()
    if content:
        raw_text +=content

In [12]:
import os
cassio.init(token=os.getenv("ASTRA_DB_APPLICATION_TOKEN") , database_id=os.getenv("ASTRA_DB_ID"))


In [15]:
from langchain_ollama import OllamaEmbeddings, OllamaLLM
embedding = OllamaEmbeddings(model="nomic-embed-text")
llm = OllamaLLM(model="llama3.2")

In [16]:
astra_vector_store= Cassandra(
    embedding=embedding,
    table_name='qa_mini_demo',
    session=None,
    keyspace=None
)

In [22]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document

raw_text = """
A Comprehensive Overview of Large Language Models
Humza Naveed, Asad Ullah Khan, Shi Qiu, Muhammad Saqib...
Abstract
Large Language Models (LLMs) have recently demonstrated remarkable capabilities in natural language processing tasks...
"""

text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

texts = text_splitter.create_documents([raw_text])

print(len(texts))
print(texts[0].page_content[:500])

1
A Comprehensive Overview of Large Language Models
Humza Naveed, Asad Ullah Khan, Shi Qiu, Muhammad Saqib...
Abstract
Large Language Models (LLMs) have recently demonstrated remarkable capabilities in natural language processing tasks...


In [23]:
astra_vector_store.add_documents(texts)
print("Inserted %i PDF chunks." % len(texts))

Inserted 1 PDF chunks.


In [24]:
from langchain_classic.indexes.vectorstore import VectorStoreIndexWrapper

astra_vector_index = VectorStoreIndexWrapper(
    vectorstore=astra_vector_store
)

In [2]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llama3.2")

In [ ]:
response = llm.invoke("What is a Large Language Model? Answer in simple words.")
print(response)